In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

### Problem

In [6]:
# Your current code
dims = 256
def get_sinusoidal_emb(t, device):
    half_dim = dims // 2
    emb = math.log(10000) / (half_dim - 1)
    emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
    emb = t[:, None] * emb[None, :]          # ← scalar t broadcast
    emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
    return emb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
t = torch.tensor([10]).to(device)

t_emb_raw = get_sinusoidal_emb(t, device)
print(t_emb_raw.shape)

torch.Size([1, 256])


In [7]:
import torch
import torch.nn as nn
import math
from einops import rearrange
import torch.nn.functional as F
import os 
from mamba_ssm import Mamba

In [8]:
class PhysConvNeXtBlock(nn.Module):
    """
    ConvNeXt V2 Block: Best for local textures and edges.
    Includes Adaptive Layer Norm (AdaLN) for Time Embedding injection.
    """
    def __init__(self, dim, mult=2):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.pwconv1 = nn.Linear(dim, 4 * dim) 
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.gamma = nn.Parameter(1e-6 * torch.ones((dim)), requires_grad=True)

    def forward(self, x, t_emb=None):
        inp = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1)
        
        if t_emb is not None:
            x = self.norm(x)
            scale, shift = t_emb.chunk(2, dim=1)
            x = x * (1 + scale.unsqueeze(1).unsqueeze(1)) + shift.unsqueeze(1).unsqueeze(1)
        else:
            x = self.norm(x)
            
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        x = self.gamma * x
        x = x.permute(0, 3, 1, 2)
        return inp + x


In [9]:
class LocalFeatureExtractor(nn.Module):
    """ 
    Adaptive Parallel Branch.
    Uses Inverted Bottleneck (Expand -> Depthwise -> Project).
    Automatically calculates padding to keep spatial dimensions constant.
    """
    def __init__(self, dim, kernel_size=3, expansion_factor=2, dilation=2):
        super().__init__()
        
        hidden_dim = int(dim * expansion_factor)
        
        # Dynamic Padding Calculation:
        # P = (dilation * (kernel_size - 1)) / 2
        # This ensures the output size equals the input size.
        padding = (dilation * (kernel_size - 1)) // 2
        
        self.net = nn.Sequential(
            # 1. Pointwise Expansion
            nn.Conv2d(dim, hidden_dim, kernel_size=1),
            nn.GELU(),
            
            # 2. Adaptive Depthwise Conv
            nn.Conv2d(hidden_dim, hidden_dim, 
                      kernel_size=kernel_size, 
                      padding=padding, 
                      dilation=dilation,
                      groups=hidden_dim), # Depthwise
            nn.GELU(),
            
            # 3. Pointwise Projection
            nn.Conv2d(hidden_dim, dim, kernel_size=1)
        )

    def forward(self, x):
        return self.net(x)


In [10]:
class PhysBiMambaBlock(nn.Module):
    """
    Bidirectional Mamba Block (BiMamba)
    Scans the image Forward AND Backward so the top-left pixel
    can 'see' the bottom-right pixel.
    """
    def __init__(self, dim, dropout = 0.05):
        super().__init__()
        self.norm = nn.LayerNorm(dim)

        # --- Horizontal Mamba -----
        self.mamba_h_fwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.mamba_h_bwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)

        # --- Vertical Mamba ---
        self.mamba_v_fwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.mamba_v_bwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        
        # Fuses Fwd+Bwd direction
        self.fusion_linear = nn.Linear(dim * 4, dim)

        self.local_conv = LocalFeatureExtractor(dim, 
                                                kernel_size=3, 
                                                dilation=1)
        
        # Optional: A Gate to let the network choose emphasis
        self.mixer = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.Sigmoid()
        )

        self.out_proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, t_emb=None):
        B, C, H, W = x.shape
        residual = x
        
        x_flat = x.flatten(2).transpose(1, 2)
        x_norm = self.norm(x_flat)

        if t_emb is not None:
            scale, shift = t_emb.chunk(2, dim=1)
            x_norm = x_norm * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

        # ---------------------------------------------------------
        # 2. HORIZONTAL SCANS (Raster Order)
        # ---------------------------------------------------------
        # Forward ->
        out_h_fwd = self.mamba_h_fwd(x_norm)
        
        # Backward <-
        x_flip = torch.flip(x_norm, dims=[1])
        out_h_bwd = self.mamba_h_bwd(x_flip)
        out_h_bwd = torch.flip(out_h_bwd, dims=[1]) # Flip back

        # ---------------------------------------------------------
        # 3. VERTICAL SCANS (Column-Major Order)
        # ---------------------------------------------------------
        # Reshape to Image -> Transpose (Swap H and W) -> Flatten
        # Result: (B, W*H, C). Now 'neighbors' in seq are vertical neighbors.
        x_v_img = x_norm.view(B, H, W, C).permute(0, 2, 1, 3) 
        x_v_flat = x_v_img.flatten(1, 2)
        
        # Down v
        out_v_fwd = self.mamba_v_fwd(x_v_flat)
        
        # Up ^
        x_v_flip = torch.flip(x_v_flat, dims=[1])
        out_v_bwd = self.mamba_v_bwd(x_v_flip)
        out_v_bwd = torch.flip(out_v_bwd, dims=[1])
        
        # Un-Transpose Vertical Outputs back to Horizontal Order
        # (B, W*H, C) -> (B, W, H, C) -> (B, H, W, C) -> (B, L, C)
        out_v_fwd = out_v_fwd.view(B, W, H, C).permute(0, 2, 1, 3).flatten(1, 2)
        out_v_bwd = out_v_bwd.view(B, W, H, C).permute(0, 2, 1, 3).flatten(1, 2)
        
        ## ---------------------------------------------------------
        # 4. Global Fusion
        # ---------------------------------------------------------
        # Combine all 4 views of the image
        global_feat = self.fusion_linear(
            torch.cat([out_h_fwd, out_h_bwd, out_v_fwd, out_v_bwd], dim=-1)
        )

        # ---------------------------------------------------------
        # 5. Local Branch (Conv)
        # ---------------------------------------------------------
        # Reshape for Conv2d
        x_img_norm = x_norm.transpose(1, 2).view(B, C, H, W)
        local_feat = self.local_conv(x_img_norm)
        local_feat = local_feat.flatten(2).transpose(1, 2)

        
        # ---------------------------------------------------------
        # 6. Gated Output
        # ---------------------------------------------------------
        combined = torch.cat([global_feat, local_feat], dim=-1)
        z = self.mixer(combined)
        
        fused = global_feat * z + local_feat * (1 - z)
        
        x_out = self.out_proj(fused)
        
        # Reshape to (B, C, H, W) for residual add
        x_out = x_out.transpose(1, 2).view(B, C, H, W)
        x_out = self.dropout(x_out)
        
        return residual + x_out


In [11]:
class GatedFusion(nn.Module):
    """
    Standard Spatial Gating (Version 1).
    Decides 'where' to fuse information pixel-by-pixel.
    """
    def __init__(self, dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(dim * 2, 1, 1),
            nn.Sigmoid()
        )
        self.out_conv = nn.Conv2d(dim, dim, 1)

    def forward(self, dec_feat, enc_feat):
        # Concatenate and calculate spatial map (B, 1, H, W)
        gate = self.conv(torch.cat([dec_feat, enc_feat], dim=1))
        # Weighted sum based on spatial location
        fused = dec_feat * (1 - gate) + enc_feat * gate
        return self.out_conv(fused)

# --- VERSION 2 COMPONENTS (SOTA) ---
class CAGatedFusion(nn.Module):
    """
    Channel Attention Gating (Version 2).
    Decides 'what features' (texture vs fog) to fuse using Global Context.
    """
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),          # Squeeze (Global Context)
            nn.Conv2d(dim * 2, dim // 2, 1),  # Compress
            nn.ReLU(inplace=True),
            nn.Conv2d(dim // 2, dim * 2, 1),  # Excite
            nn.Sigmoid()                      # Weight
        )
        self.conv = nn.Conv2d(dim, dim, 1)

    def forward(self, dec_feat, enc_feat):
        combined = torch.cat([dec_feat, enc_feat], dim=1)
        weights = self.attn(combined)
        w_dec, w_enc = weights.chunk(2, dim=1)
        # Channel-wise weighted fusion
        fused = (dec_feat * w_dec) + (enc_feat * w_enc)
        return self.conv(fused)



class PixelShuffleUpsample(nn.Module):
    """
    SOTA Trick: Replaces ConvTranspose2d to eliminate checkerboard artifacts.
    """
    def __init__(self, dim_in, dim_out):
        super().__init__()
        # We need to project to (dim_out * 4) so PixelShuffle(2) results in dim_out
        self.conv = nn.Conv2d(dim_in, dim_out * 4, 3, 1, 1)
        self.pixel_shuffle = nn.PixelShuffle(2) # Scale x2
        
    def forward(self, x):
        return self.pixel_shuffle(self.conv(x))

- **Scalar t only (no spatial/physics link)**: The entire image gets the same global timestep even though haze is highly non-homogeneous. Flat walls and dense smoke regions should evolve at different “speeds”.
- **No interaction with your physics branch**: You already predict local t_map and A, but t_vec is injected independently. The velocity field never “knows” the local haze density when deciding how much to move.
- **Fixed sinusoidal schedule**: Classic diffusion-style embedding assumes uniform difficulty across time. In flow-matching with small NFE=10, early steps (thick haze) need stronger guidance.
- **Simple MLP injection**: t_vec is added via basic AdaLN in PhysConvNeXtBlock, but no fusion with Mamba’s bi-directional features or density map.
  **No density-aware modulation**: Your Density-Aware Flow Loss exists, but the conditioning signal itself is not density-aware.

*Note*: The model is too generic for the physics-grounded dehazing tasks => Biggest remaining gaps

Easiest Fix: Add the Physics-Aware information to Time Embedding

In [12]:
class FM_PhysMamba_UNET(nn.Module):
    def __init__(self, base_dim, dim_mults, time_dim_mult, enc_blocks_list, 
                          dec_blocks_list, physics_guided = True):
        super().__init__()
        
        self.physics_guided = physics_guided
        self.dims = [base_dim * m for m in dim_mults]

        num_mid_blocks = enc_blocks_list[-1] if len(enc_blocks_list) >= len(self.dims) else 2

        # --- Time & Physics Embedding ---
        time_dim = base_dim * time_dim_mult
        self.time_mlp = nn.Sequential(
            nn.Linear(base_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )
        
        self.phys_gate_fusion = nn.Sequential(
            nn.Linear(time_dim + 5, time_dim),     # 5 = global_A(3) + avg_t(1) + density(1)
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

        self.down_time_projs = nn.ModuleList()
        self.up_time_projs = nn.ModuleList()

        # --- ENCODER ---
        self.init_conv = nn.Conv2d(in_channels, self.dims[0], 3, 1, 1)
        self.downs = nn.ModuleList()
        self.downsamples = nn.ModuleList()
        
        for i in range(len(self.dims) - 1):
            dim_in, dim_out = self.dims[i], self.dims[i+1]
            self.down_time_projs.append(nn.Linear(time_dim, dim_in * 2))
            
            # Use ModuleList instead of Sequential for checkpointing control
            blocks = nn.ModuleList([PhysConvNeXtBlock(dim_in)])
            num_mamba = enc_blocks_list[i] if i < len(enc_blocks_list) else 1
            for _ in range(num_mamba):
                blocks.append(PhysBiMambaBlock(dim_in))
            
            self.downs.append(blocks)
            self.downsamples.append(nn.Conv2d(dim_in, dim_out, 4, 2, 1)) 


        mid_dim = self.dims[-1]
        self.mid_time_proj = nn.Linear(time_dim, mid_dim * 2)
        self.mid_blocks = nn.ModuleList()
        for _ in range(num_mid_blocks):
            self.mid_blocks.append(PhysBiMambaBlock(mid_dim))
        
        self.atm_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(mid_dim, 64), nn.SiLU(),
            nn.Linear(64, 3), nn.Sigmoid() 
        )

        # --- DECODER ---
        self.ups = nn.ModuleList()
        self.up_samples = nn.ModuleList()
        self.gates = nn.ModuleList()
        
        for idx, i in enumerate(range(len(self.dims)-2, -1, -1)):
            dim_in, dim_out = self.dims[i+1], self.dims[i]
            self.up_time_projs.append(nn.Linear(time_dim, dim_out * 2))
            # self.up_samples.append(nn.ConvTranspose2d(dim_in, dim_out, 2, 2))
            # self.gates.append(GatedFusion(dim_out))

            # --- SWITCHING LOGIC ---
            if self.use_version == 1:
                # Version 1: Standard Deconv + Spatial Gating
                self.up_samples.append(nn.ConvTranspose2d(dim_in, dim_out, 2, 2))
                self.gates.append(GatedFusion(dim_out))
            else:
                # Version 2: PixelShuffle + Channel Attention Gating (SOTA)
                self.up_samples.append(PixelShuffleUpsample(dim_in, dim_out))
                self.gates.append(CAGatedFusion(dim_out))
            # -----------------------
            
            layers = nn.ModuleList()
            num_mamba = dec_blocks_list[i] if i < len(dec_blocks_list) else 1
            for _ in range(num_mamba):
                if i > 0: layers.append(PhysBiMambaBlock(dim_out))
                else: layers.append(PhysConvNeXtBlock(dim_out))
            layers.append(PhysConvNeXtBlock(dim_out)) 
            self.ups.append(layers)

        self.trans_head = nn.Sequential(
            nn.Conv2d(self.dims[0], 16, 3, 1, 1), nn.SiLU(),
            nn.Conv2d(16, 1, 1), nn.Sigmoid() 
        )
        self.final_conv = nn.Conv2d(self.dims[0], 3, 1)
        nn.init.constant_(self.trans_head[-2].bias, 1.0)

    def get_sinusoidal_emb(self, t, device):
        half_dim = self.dims[0] // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb
        
    def get_time_embedding(self, t_scalar, local_t_map, local_A, density_map):
        """
        t_scalar: [B]   (your original timestep, 0~1)
        local_t_map, local_A, density_map: [B, 1/3, H, W]
        """
        B = t_scaler.shape[0]

        # 1. Classic sinusoidal (global schedule)
        t_emb = self.get_sinusoidal_emb(t_scalar, t_scalar.device)    
        t_vec = self.time_mlp(t_emb)                                  # [B, time_dim]

        # 2. Extract physics statistics (global context)
        avg_t = local_t_map.mean(dim = (2, 3), keepdim = False)      # [B,1]
        avg_A = local_A.mean(dim=(2,3), keepdim=False)               # [B,3]
        avg_density = density_map.mean(dim=(2,3), keepdim=False)     # [B,1]

        phys_stats = torch.cat([avg_A, avg_t, avg_density], dim=1)   # [B,5]

        # 3. Fuse scalar time + physics
        fused = torch.cat([t_vec, phys_stats], dim=1)                # [B, time_dim+5]
        t_vec_phys = self.phys_time_fusion(fused)                    # final conditioning vector

        return t_vec_phys
    
    def forward(self, x, t_scalar, local_t_map=None, local_A=None, density_map=None):
        if local_t_map is None:
            local_t_map = torch.ones_like(x[:,:1]) * 0.5   # fallback
            
        t_vec = self.get_time_embedding(t_scalar, local_t_map, local_A, density_map)
        
        h = self.init_conv(x)
        skips = []
        
        # 2. ENCODER
        for i, (block_list, down_layer) in enumerate(zip(self.downs, self.downsamples)):
            t_emb = self.down_time_projs[i](t_vec)
            for layer in block_list:
                if self.use_checkpoint and self.training:
                    h = checkpoint.checkpoint(layer, h, t_emb, use_reentrant=False)
                else:
                    h = layer(h, t_emb)
            skips.append(h)
            h = down_layer(h)
            
        # 3. BOTTLENECK
        t_emb_mid = self.mid_time_proj(t_vec)

        # We will adapt to this later (Maybe for the O-HAZE DENSE-HAZE Training)
        for block in self.mid_blocks:
            if self.use_checkpoint and self.training:
                h = checkpoint.checkpoint(block, h, t_emb_mid, use_reentrant=False)
            else:
                h = block(h, t_emb_mid)
        
        A_pred = self.atm_head(h).view(-1, 3, 1, 1)
        
        phys_cond = torch.cat([t_vec, A_pred.squeeze(-1).squeeze(-1)], dim=-1)
        t_vec = self.phys_gate(phys_cond)

        # 4. DECODER
        for i in range(len(self.ups)):
            h = self.up_samples[i](h) 
            if len(skips) > 0:
                skip = skips.pop()
                h = self.gates[i](h, skip)
            
            block_list = self.ups[i]
            t_emb_dec = self.up_time_projs[i](t_vec)
            
            for layer in block_list:
                if self.use_checkpoint and self.training:
                    h = checkpoint.checkpoint(layer, h, t_emb_dec, use_reentrant=False)
                else:
                    h = layer(h, t_emb_dec)
                    
        t_map = self.trans_head(h)
        if self.physics_guided:
            h = h * (1 + t_map)
            
        v_pred = self.final_conv(h)

        return v_pred, t_map, A_pred

### Maybe seperating to different stages can be fine

**Stage 1**: Physics Estimator

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math

def get_pad_layer(pad_type):
    if(pad_type in ['refl','reflect']):
        PadLayer = nn.ReflectionPad2d
    elif(pad_type in ['repl','replicate']):
        PadLayer = nn.ReplicationPad2da
    elif(pad_type=='zero'):
        PadLayer = nn.ZeroPad2d
    else:
        print(f'Pad type [{pad_type}] not recognized')
    return PadLayer


class AntiAlias_Downsample(nn.Module):
    def __init__(self, channels, pad_type = 'reflect', filt_size = 3, 
                        stride = 2, pad_off = 0):
        super(AntiAlias_Downsample, self).__init__()
        self.filt_size = filt_size
        self.pad_off = pad_off
        self.pad_type = pad_type

        # Asymmetric padding (round up at the top and round down at the bottom)
        # Perfect when kernel size is 2 
        self.pad_sizes = [int(1. * (filt_size - 1) / 2), int(np.ceil(1. * (filt_size - 1) / 2)),
                          int(1. * (filt_size - 1) / 2), int(np.ceil(1. * (filt_size - 1) / 2))]
        self.pad_sizes = [pad_size + pad_off for pad_size in self.pad_sizes]
        self.stride = stride 
        self.off = int((self.stride - 1) / 2.)
        self.channels = channels 

        # Define the binomial filter weights
        if(self.filt_size==1):
            a = np.array([1.,])
        elif(self.filt_size==2):
            a = np.array([1., 1.])
        elif(self.filt_size==3):
            a = np.array([1., 2., 1.])
        elif(self.filt_size==4):    
            a = np.array([1., 3., 3., 1.])
        elif(self.filt_size==5):    
            a = np.array([1., 4., 6., 4., 1.])
        elif(self.filt_size==6):    
            a = np.array([1., 5., 10., 10., 5., 1.])
        elif(self.filt_size==7):    
            a = np.array([1., 6., 15., 20., 15., 6., 1.])
            
        # Create a 2D filter by taking the outer product of the 1D filter
        filt = torch.tensor(a[:, None] * a[None, :], dtype = torch.float32)
        filt = filt / torch.sum(filt) # Normalize

        # Reshape to (out_channels, in_channels/groups, kH, kW) for 
        # depthwise convolution
        filt = filt.view(1, 1, filt_size, filt_size)
        filt = filt.repeat(channels, 1, 1, 1)

        # Register as a buffer so PyTorch knows these are NOT trainable parameters
        self.register_buffer('filt', filt)
        self.pad = get_pad_layer(pad_type)(self.pad_sizes)

    def forward(self, inp):
        if (self.filt_size == 1):
            if (self.pad_off == 0):
                return inp[:, :, ::self.stride, ::self.stride] 
            else:
                return self.pad(inp)[:, :, ::self.stride, ::self.stride] 

        else:
            return F.conv2d(self.pad(inp), self.filt, stride = self.stride, groups = inp.shape[1])

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [14]:
class BilinearUpsample(nn.Module):
    """
    Used by BOTH variants. Guarantees no checkerboard artifacts during decoding.
    """
    def __init__(self, dim_in, dim_out):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv = nn.Conv2d(dim_in, dim_out, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        return self.conv(self.up(x))

In [15]:
## Variant Specific Downsampling
class VariantA_StandardDownsample(nn.Module):
    """
    Standard stride=2 convolution. 
    Prone to aliasing and shift-variance (ignores Nyquist theorem).
    """
    def __init__(self, dim_in, dim_out):
        super().__init__()
        # Strided convolution drops 75% of pixels abruptly
        self.down = nn.Conv2d(dim_in, dim_out, kernel_size=4, stride=2, padding=1)

    def forward(self, x):
        return self.down(x)

In [17]:
class VariantB_AntiAliasedDownsample(nn.Module):
    """
    Anti-Aliased Downsampling (BlurPool) based on Richard Zhang's paper.
    Preserves shift-invariance and prevents high-frequency aliasing.
    """
    def __init__(self, dim_in, dim_out):
        super().__init__()
        # 1. Feature Mixing (Stride 1 preserves all spatial information)
        self.conv = nn.Conv2d(dim_in, dim_out, kernel_size=3, stride=1, padding=1)
        # 2. Anti-aliased spatial reduction (Low-pass filter + subsampling)
        # NOTE: Make sure your AntiAlias_Downsample class is defined in the script!
        self.aa_down = AntiAlias_Downsample(channels=dim_out, filt_size=3, stride=2)

    def forward(self, x):
        return self.aa_down(self.conv(x))

In [18]:
class AblationPhysicsEstimator(nn.Module):
    """
    Stage 1: Configurable Physics Estimator.
    Tests Standard Downsampling vs Anti-Aliased Downsampling.
    Both use Bilinear Upsampling to isolate the effect.
    """
    def __init__(self, variant='B', in_channels=3, base_dim=32):
        super().__init__()
        self.variant = variant
        
        # --- INITIAL ENCODER ---
        self.init_conv = nn.Conv2d(in_channels, base_dim, kernel_size=3, padding=1)
        self.enc1 = nn.Conv2d(base_dim, base_dim, kernel_size=3, padding=1)

        # ==========================================
        # --- ABLATION SWITCH LOGIC (DOWNSAMPLING)
        # ==========================================
        if self.variant == 'A':
            print("Initializing Stage 1 with Variant A (Standard Stride=2 Downsampling)")
            self.down1 = VariantA_StandardDownsample(base_dim, base_dim * 2)
            self.down2 = VariantA_StandardDownsample(base_dim * 2, base_dim * 4)
        elif self.variant == 'B':
            print("Initializing Stage 1 with Variant B (Anti-Aliased Downsampling)")
            self.down1 = VariantB_AntiAliasedDownsample(base_dim, base_dim * 2)
            self.down2 = VariantB_AntiAliasedDownsample(base_dim * 2, base_dim * 4)
        else:
            raise ValueError("Variant must be 'A' or 'B'")


        self.enc2 = nn.Conv2d(base_dim * 2, base_dim * 2, kernel_size=3, padding=1)
        
        # --- BOTTLENECK ---
        self.bottleneck = nn.Conv2d(base_dim * 4, base_dim * 4, kernel_size=3, padding=1)
        
        # --- GLOBAL ATMOSPHERIC LIGHT (A) HEAD ---
        self.A_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(base_dim * 4, 3), nn.Sigmoid() 
        )
        
        # --- DECODER (SHARED: Bilinear Upsampling for BOTH variants) ---
        self.up1 = BilinearUpsample(base_dim * 4, base_dim * 2)
        self.dec1_conv = nn.Conv2d(base_dim * 4, base_dim * 2, kernel_size=1) 
        self.dec1 = nn.Conv2d(base_dim * 2, base_dim * 2, kernel_size=3, padding=1)
        
        self.up2 = BilinearUpsample(base_dim * 2, base_dim)
        self.dec2_conv = nn.Conv2d(base_dim * 2, base_dim, kernel_size=1)
        self.dec2 = nn.Conv2d(base_dim, base_dim, kernel_size=3, padding=1)
        
        # --- TRANSMISSION MAP (t) HEAD ---
        self.t_head = nn.Sequential(
            nn.Conv2d(base_dim, 1, kernel_size=3, padding=1),
            nn.Sigmoid() 
        )


    def forward(self, x):
        # Encode
        e1 = self.enc1(self.init_conv(x))
        d1 = self.down1(e1)
        
        e2 = self.enc2(d1)
        d2 = self.down2(e2)
        
        # Bottleneck
        b = self.bottleneck(d2)
        A = self.A_head(b).view(-1, 3, 1, 1)
        
        # Decode
        u1 = self.up1(b)
        u1 = torch.cat([u1, e2], dim=1) # Skip connection
        u1 = self.dec1(self.dec1_conv(u1))
        
        u2 = self.up2(u1)
        u2 = torch.cat([u2, e1], dim=1) # Skip connection
        u2 = self.dec2(self.dec2_conv(u2))
        
        t_map = self.t_head(u2)
        return t_map, A

What to Look For in Your Ablation Results

Because you have correctly isolated downsampling, any differences in the output are purely due to aliasing in the encoder.

1. **Shift Equivariance Test**: If you shift the input image by 1 pixel, does the output $t(x)$ map shift exactly by 1 pixel? Variant A will likely "flicker" or change structural shapes slightly when shifted. Variant B should remain perfectly consistent.
2. **Edge Smoothness in $t(x)$**: Examine the boundaries between foreground objects and the background sky in the transmission map. Variant A might show "jagged" or "stair-stepped" edges because the stride=2 convolution arbitrarily dropped edge pixels. Variant B will have smooth, continuous gradients at the edges.
3. **Loss Spikes during Training**: You might notice that Variant B trains with fewer loss spikes. Anti-aliasing naturally regularizes the network, preventing high-frequency noise from destabilizing the gradient updates.